# 1. Energy Calibration with POSM/POSM2/POSM3 Files

**Objective:** Establish energy calibration using POSM calibration files, extract calibration parameters with uncertainties, and create final calibration curve with error bands.

## Setup and Helper Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import signal, optimize
import re

# Directory for data files
DATA_DIR = Path('.')

def load_alpha_spectrum_file(file_path):
    """Load alpha spectroscopy MCA data file.
    Returns: (channel_array, counts_array)
    """
    text = Path(file_path).read_text(encoding='utf-8', errors='ignore').splitlines()
    channels = []
    counts = []
    
    for line in text:
        # Try to parse as channel/count pair (format: channel, count)
        match = re.match(r'\s*(\d+),\s*(\d+)', line)
        if match:
            channels.append(float(match.group(1)))
            counts.append(float(match.group(2)))
    
    return np.array(channels), np.array(counts)

def calcular_centroide(canais, contagens):
    """Calcula o centróide (média ponderada) de um pico."""
    canais = np.array(canais)
    contagens = np.array(contagens)
    
    if np.sum(contagens) == 0:
        return 0, 0, 0
        
    centroid = np.sum(canais * contagens) / np.sum(contagens)
    
    # Peak width (sigma)
    variance = np.sum(contagens * (canais - centroid)**2) / np.sum(contagens)
    peak_sigma = np.sqrt(variance)
    
    # Centroid uncertainty
    N_total = np.sum(contagens)
    sigma_centroid = peak_sigma / np.sqrt(N_total)
    
    return centroid, sigma_centroid, peak_sigma

def ler_espetro(nome_ficheiro, canal_min, canal_max):
    """Lê o ficheiro .asc e extrai canais e contagens."""
    canais = []
    contagens = []
    
    try:
        with open(nome_ficheiro, 'r') as f:
            linhas = f.readlines()
            
        lendo_dados = False
        for linha in linhas:
            if "Chn" in linha and "Counts" in linha:
                lendo_dados = True
                continue
                
            if lendo_dados:
                partes = linha.split(',')
                if len(partes) >= 2:
                    try:
                        canal = int(partes[0].strip())
                        contagem = int(partes[1].strip())
                        
                        if canal_min <= canal <= canal_max:
                            canais.append(canal)
                            contagens.append(contagem)
                    except ValueError:
                        pass
    except FileNotFoundError:
        print(f"ERRO: Não foi possível encontrar '{nome_ficheiro}'.")
                        
    return canais, contagens

print('Setup complete. Helper functions loaded.')

## Calibration Analysis with POSM Files

In [ ]:
# ==========================================
# LOAD DATA FROM POSM FILES
# ==========================================
ficheiro_1 = 'POSM.ASC'
ficheiro_2 = 'POSM2.ASC'
ficheiro_3 = 'POSM3.ASC'

# Ler picos (estimate regions - adjust as needed)
canais_p1, cont_p1 = ler_espetro(ficheiro_1, 320, 360)
canais_p2, cont_p2 = ler_espetro(ficheiro_2, 320, 360)
canais_p3, cont_p3 = ler_espetro(ficheiro_3, 320, 360)

# Calculate centroids
centroide_1, sigma_1, sigma_peak_1 = calcular_centroide(canais_p1, cont_p1)
centroide_2, sigma_2, sigma_peak_2 = calcular_centroide(canais_p2, cont_p2)
centroide_3, sigma_3, sigma_peak_3 = calcular_centroide(canais_p3, cont_p3)

centroides_pulser = np.array([centroide_1, centroide_2, centroide_3])
sigma_pulser_array = np.array([sigma_1, sigma_2, sigma_3])

# Use Po-210 energy as calibration anchor
energia_po_real = 5.304  # MeV

# Assume three different measurement conditions or dial settings
# For POSM files, we treat them as three independent measurements of the same source
sinal_pulser = np.array([1.0, 2.0, 3.0])  # Arbitrary indexing for three measurements

print("="*70)
print("POSM FILES LOADED")
print("="*70)
print(f"\nFile 1 ({ficheiro_1}):")
print(f"  Centroid: {centroide_1:.2f} ± {sigma_1:.3f} channels")
print(f"\nFile 2 ({ficheiro_2}):")
print(f"  Centroid: {centroide_2:.2f} ± {sigma_2:.3f} channels")
print(f"\nFile 3 ({ficheiro_3}):")
print(f"  Centroid: {centroide_3:.2f} ± {sigma_3:.3f} channels")
print(f"\nMeasurement Indices: {sinal_pulser}")

## Weighted Linear Regression and Calibration

In [ ]:
# ==========================================
# WEIGHTED LINEAR REGRESSION
# ==========================================
print("="*70)
print("WEIGHTED LINEAR REGRESSION")
print("="*70)

# Weights: inverse of variance
weights = 1.0 / (sigma_pulser_array**2)

# Weighted mean
x_mean = np.average(sinal_pulser, weights=weights)
y_mean = np.average(centroides_pulser, weights=weights)

# Weighted covariance and variance
cov_xy = np.average((sinal_pulser - x_mean) * (centroides_pulser - y_mean), weights=weights)
var_x = np.average((sinal_pulser - x_mean)**2, weights=weights)

# Slope and intercept
m_p_weighted = cov_xy / var_x
c_zero_weighted = y_mean - m_p_weighted * x_mean

# ==========================================
# CHI-SQUARED CALCULATION
# ==========================================
fitted_centroides = m_p_weighted * sinal_pulser + c_zero_weighted
residuos_weighted = centroides_pulser - fitted_centroides

chi_squared = np.sum((residuos_weighted**2) / fitted_centroides)
dof = len(sinal_pulser) - 2
reduced_chi_sq = chi_squared / dof

print(f"\nResiduals:")
for i, (obs, fit, res) in enumerate(zip(centroides_pulser, fitted_centroides, residuos_weighted)):
    chi2_term = (res**2) / fit
    print(f"  Point {i+1}: obs={obs:.3f}, fitted={fit:.3f}, residual={res:.6f}, χ²_term={chi2_term:.6f}")

print(f"\nχ² = {chi_squared:.6f}")
print(f"Reduced χ² = {reduced_chi_sq:.6f}")
print(f"DOF = {dof}")

# Uncertainties in slope and intercept
sigma_m_sq = 1.0 / np.sum(weights * (sinal_pulser - x_mean)**2)
sigma_m = np.sqrt(sigma_m_sq)

sigma_c_sq = sigma_m_sq * np.sum(weights * sinal_pulser**2) / np.sum(weights)
sigma_c = np.sqrt(sigma_c_sq)

# R² for weighted fit
ss_res = np.sum(weights * residuos_weighted**2)
ss_tot = np.sum(weights * (centroides_pulser - y_mean)**2)
r_quadrado_weighted = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0

print(f"\nSlope (m):        {m_p_weighted:.6f} ± {sigma_m:.6f}")
print(f"Intercept (c):    {c_zero_weighted:.6f} ± {sigma_c:.6f}")
print(f"R²:               {r_quadrado_weighted:.6f}")

# ==========================================
# ENERGY CALIBRATION
# ==========================================
ganho_energia = energia_po_real / (y_mean - c_zero_weighted)
sigma_ganho = (sigma_m / (y_mean - c_zero_weighted))

b_energia = -ganho_energia * c_zero_weighted
sigma_b = np.sqrt((ganho_energia * sigma_c)**2 + (c_zero_weighted * sigma_ganho)**2)

print(f"\nEnergy Calibration:")
print(f"  Gain (m):       {ganho_energia:.6f} ± {sigma_ganho:.6f} MeV/channel")
print(f"  Intercept (b):  {b_energia:.6f} ± {sigma_b:.6f} MeV")
print(f"\nCalibration Equation: E(C) = {ganho_energia:.6f} × C + {b_energia:.6f}")

In [ ]:
# ==========================================
# PROPAGATE TO ENERGY SPACE
# ==========================================
energias_pulser_calibradas = ganho_energia * (centroides_pulser - c_zero_weighted)

sigma_pulser_energia = np.sqrt((sigma_ganho * (centroides_pulser - c_zero_weighted))**2 + 
                                (ganho_energia * sigma_pulser_array)**2 + 
                                sigma_b**2)

print(f"\n--- MEASUREMENTS IN ENERGY SPACE ---")
for i, (ch, E, sig_ch, sig_E) in enumerate(zip(centroides_pulser, energias_pulser_calibradas, 
                                                  sigma_pulser_array, sigma_pulser_energia), 1):
    print(f"  Measurement {i}: {ch:.2f} ± {sig_ch:.3f} ch  →  {E:.4f} ± {sig_E:.4f} MeV")

print(f"\n--- PO-210 REFERENCE ---")
print(f"  Energy: {energia_po_real} MeV")

## Final Calibration Plot

In [ ]:
# ==========================================
# FINAL ENERGY CALIBRATION PLOT
# ==========================================

fig, ax = plt.subplots(figsize=(13, 8))

# Plot range
canais_plot = np.linspace(0, 1024, 500)

# Central calibration line: E = m * C + b
energias_plot = ganho_energia * (canais_plot - c_zero_weighted)

# Uncertainty bands
sigma_E_plot = np.sqrt((sigma_ganho * (canais_plot - c_zero_weighted))**2 + 
                        (ganho_energia * sigma_c)**2 + 
                        sigma_b**2)

# Plot 1-sigma uncertainty band
ax.fill_between(canais_plot, energias_plot - sigma_E_plot, energias_plot + sigma_E_plot, 
                 alpha=0.3, color='blue', label='±1σ uncertainty band', zorder=1)

# Plot calibration line
ax.plot(canais_plot, energias_plot, 'b-', linewidth=3, label='Calibration curve', zorder=3)

# Plot measurement points with FULL error bars (X and Y)
ax.errorbar(centroides_pulser, energias_pulser_calibradas, 
            yerr=sigma_pulser_energia, xerr=sigma_pulser_array,
            fmt='o', color='darkblue', markersize=11, capsize=7, capthick=2.5, elinewidth=2.5,
            label='POSM measurements', zorder=4)

# Plot Po-210 reference point
ax.axhline(energia_po_real, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Po-210 anchor: {energia_po_real} MeV', zorder=2)

# Formatting
ax.set_xlabel('Channel (MCA)', fontsize=13, fontweight='bold')
ax.set_ylabel('Energy (MeV)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1024)
ax.set_ylim(-0.5, 7)
ax.grid(True, linestyle=':', alpha=0.5, zorder=0)
ax.legend(fontsize=11, loc='upper left', framealpha=0.98)

# Calibration parameters box
textstr = f'Energy Calibration Parameters:\n\n' \
          f'E(C) = m·C + b\n\n' \
          f'm (slope):  {ganho_energia:.6f} ± {sigma_ganho:.6f} MeV/ch\n' \
          f'b (intercept): {b_energia:.6f} ± {sigma_b:.6f} MeV\n' \
          f'C₀ (channel offset): {c_zero_weighted:.2f} ch\n\n' \
          f'χ² (reduced): {reduced_chi_sq:.4f}\n' \
          f'R²: {r_quadrado_weighted:.6f}'

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black', linewidth=1.5)
ax.text(0.98, 0.50, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right', bbox=props, family='monospace')

plt.tight_layout()
plt.show()

print("="*70)
print("FINAL ENERGY CALIBRATION (POSM FILES)")
print("="*70)
print(f"\nCalibration Equation:")
print(f"  E(C) = {ganho_energia:.6f} × C + {b_energia:.6f} MeV")
print(f"\nCalibration Parameters:")
print(f"  Gain (m):          {ganho_energia:.6f} ± {sigma_ganho:.6f} MeV/channel")
print(f"  Intercept (b):     {b_energia:.6f} ± {sigma_b:.6f} MeV")
print(f"  Channel offset:    {c_zero_weighted:.2f} channels")
print(f"\nGoodness of Fit:")
print(f"  χ² (reduced):      {reduced_chi_sq:.4f}")
print(f"  R²:                {r_quadrado_weighted:.6f}")
print(f"\nPOSM Measurements (3):")
for i, (ch, E, sig_ch, sig_E) in enumerate(zip(centroides_pulser, energias_pulser_calibradas, 
                                                  sigma_pulser_array, sigma_pulser_energia), 1):
    print(f"  Measurement {i}: {ch:.2f} ± {sig_ch:.3f} ch  →  {E:.4f} ± {sig_E:.4f} MeV")
print(f"\nPo-210 Calibration Anchor: {energia_po_real} MeV")